In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_assignment_whl(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Assignment"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "whl",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_optiverse",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== main_assignment ==================

        main_assignment_extra_filters = None

        if main_assignment_extra_filters is not None:
            main_assignment_extra_filters.append(date_filter)
            main_assignment_final_filters = " AND ".join(main_assignment_extra_filters)
        else:
            main_assignment_final_filters = date_filter

        main_assignment_selected_fields = [
            "CDI",
            "PortfolioID",
            "TaxNumber",
            "Unit",
            "Officer",
            "Supervisor",
            "Manager",
            "Head",
            "AssistantManager",
            "businessdate",
        ]

        main_assignment = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_mainassignment",  # check name
            main_assignment_final_filters,
            main_assignment_selected_fields,
        )

        # ================== Join Conditions==================

        main_assignment_portfolio_join_condition = (
            main_assignment["PortfolioID"] == portfolio["portfolio_id_optiverse"]
        )

        # Joins
        target_table_whl = main_assignment.join(
            portfolio,
            main_assignment_portfolio_join_condition,
            "left",
        )

        # HashKeys
        main_assignment_AssignmentId_primary_key_fields = [
            main_assignment.CDI,
            lit("whl"),
        ]

        # Final Table
        target_table_whl = target_table_whl.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(";", *main_assignment_AssignmentId_primary_key_fields),
                256,
            ),
        )

        # ================== Selection of Final Columns ==================

        result_df_whl = target_table_whl.select(
            target_table_whl.AssignmentId.cast("String").alias("AssignmentId"),
            lit("whl").cast("String").alias("SourceSystem"),
            target_table_whl.PortfolioID.cast("String").alias("PortfolioKey"),
            target_table_whl.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_whl.CDI.cast("String").alias("AssignmentKey"),
            lit(None).cast("Int").alias("CaseCode"),
            lit(None).cast("String").alias("ExposureKey"),
            lit(None).cast("String").alias("PartyKey"),
            target_table_whl.CDI.cast("String").alias("CDI"),
            target_table_whl.TaxNumber.cast("String").alias("TaxIdentificationNumber"),
            lit(None).cast("String").alias("AgreementKey"),
            lit(None).cast("String").alias("InternalAssignmentStartDate"),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            lit(None).cast("String").alias("ExternalCompanyPartyKey"),
            lit(None).cast("String").alias("ExternalCompanyFullName"),
            lit(None).cast("String").alias("AssignmentTypeCode"),
            lit(None).cast("String").alias("AssignmentType"),
            target_table_whl.Unit.cast("String").alias("TeamCode"),
            target_table_whl.Unit.cast("String").alias("Team"),
            target_table_whl.Officer.cast("String").alias("AgentPartyKey"),
            target_table_whl.Officer.cast("String").alias("AgentFullName"),
            target_table_whl.Officer.cast("String").alias("AgentLoginName"),
            target_table_whl.Supervisor.cast("String").alias("Supervisor"),
            target_table_whl.AssistantManager.cast("String").alias("AssistantManager"),
            target_table_whl.Manager.cast("String").alias("Manager"),
            target_table_whl.Head.cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            lit(None).cast("String").alias("ProcessType"),
            lit(None).cast("Int").alias("PacketCode"),
            lit(None).cast("String").alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            lit(None).cast("timestamp").alias("PacketAssignmentDate"),
            lit(None).cast("Int").alias("PaccCode"),
            lit(None).cast("Int").alias("QueCode"),
            lit(None).cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            lit(None).cast("double").alias("OutstandingAmount"),
            lit(None).cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            main_assignment.businessdate.alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),   
        )


        # ======================Append ===================================

        result_df_whl.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

        logging.info("We did it!")



In [0]:
load_curated_assignment_whl(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)